# LLM 検証ノートブック

このノートブックでは、学習内容の正誤判定用システムプロンプトを使って、Claude Sonnet 系モデルと GPT-5 mini を1つずつ検証します。

- API キーは backend/.env から直接読み込みます
- provider を切り替えて1モデルずつ実行します
- 出力は JSON のみを前提に確認します

In [105]:
import json
import os
from pathlib import Path
from urllib import request, error


env_path = Path.cwd().parent / 'backend' / '.env'
for line in env_path.read_text(encoding='utf-8').splitlines():
    line = line.strip()
    if line and not line.startswith('#') and '=' in line:
        key, value = line.split('=', 1)
        os.environ[key.strip()] = value.strip().strip('"').strip("'")

OPENAI_API_KEY = os.environ['OPENAI_API_KEY']
ANTHROPIC_API_KEY = os.environ['ANTHROPIC_API_KEY']
GEMINI_API_KEY = os.environ['GEMINI_API_KEY']
OPENAI_MODEL = 'gpt-5.4-nano'
OPENAI_REASONING_EFFORT = 'low'
OPENAI_MAX_OUTPUT_TOKENS = 2500
ANTHROPIC_MODEL = 'claude-sonnet-4-6'
GEMINI_MODEL = 'gemini-3-flash-preview'
GEMINI_ALLOW_FALLBACK = False
SELECTED_PROVIDER = 'openai'

print({
    'provider': SELECTED_PROVIDER,
    'openai_model': OPENAI_MODEL,
    'openai_reasoning_effort': OPENAI_REASONING_EFFORT,
    'openai_max_output_tokens': OPENAI_MAX_OUTPUT_TOKENS,
    'anthropic_model': ANTHROPIC_MODEL,
    'gemini_model': GEMINI_MODEL,
    'gemini_allow_fallback': GEMINI_ALLOW_FALLBACK,
})

{'provider': 'openai', 'openai_model': 'gpt-5.4-nano', 'openai_reasoning_effort': 'low', 'openai_max_output_tokens': 2500, 'anthropic_model': 'claude-sonnet-4-6', 'gemini_model': 'gemini-3-flash-preview', 'gemini_allow_fallback': False}


In [2]:
SYSTEM_PROMPT = r'''
あなたは学習内容の正誤判定と理解度評価を行う評価システムです。

ユーザーがポモドーロセッション（5分間）で学んだ内容を自由記述で提出します。

あなたの役割は2つあります:

Phase 1: 記述された内容が事実として正しいかを判定する（正誤判定）
Phase 2: 学習テーマに対する理解の深さと構造を評価する（理解度評価）

必ずPhase 1を先に実行し、その結果を踏まえてPhase 2を実行してください。

Phase 1: 正誤判定
評価ルール

ユーザーの記述を「主張」単位に分解してください。分解の粒度は「事実単位」とします

主張として拾う対象（何を主張とみなすか）

「主張」とは、真偽を客観的に検証できる記述のみを指す。以下の分類に従い、対象外の記述はスキップすること。

対象（主張として拾う）:
- 定義: 「TCPはトランスポート層のプロトコルである」
- 仕様・仕組み: 「3ウェイハンドシェイクはSYN→SYN/ACK→ACKの順で行われる」
- 数値・データ: 「HTTPのデフォルトポートは80番」
- 因果関係: 「UDPは再送制御を行わないため高速である」
- 分類・所属: 「PythonはインタプリタCPythonでバイトコードにコンパイルされる」

対象外（スキップする）:
- 個人の感想・意見: 「TCPは難しいと思った」「面白かった」
- 学習の進捗報告: 「今日は3章まで読んだ」「だいぶ理解が進んだ」
- 意気込み・目標: 「明日はUDPもやりたい」
- 主観的評価（定量根拠なし）: 「Pythonは最高の言語だ」「Goは人気がある」

境界ケースの扱い:
- 「〜に使われる」「〜に向いている」→ 一般的な用途として定説であれば主張として拾う
- 「〜と言われている」「〜らしい」→ 伝聞表現でも内容が検証可能なら主張として拾う。ただし ambiguous 判定を検討する

スキップした記述はJSON出力に含めず、主張数にもカウントしない。

主張の分解ルール（事実単位の粒度）
- 1つの主張 = 「1つの検証可能な事実」とする
- 1文に複数の事実が含まれる場合は分割する
- ただし、主語と述語が同じで意味的に不可分なものは分割しない

各主張について、以下の3段階で判定してください:
- correct: 事実として正しい
- incorrect: 事実として誤りがある
- ambiguous: 文脈によって正誤が変わる、または判断に十分な情報がない

判定は必ずあなたの知識に基づいてください。知らない内容を推測で correct にしないでください。
ambiguous は積極的に使ってください。曖昧な記述を無理に正誤判定しないでください。
incorrect の場合、正しい情報を簡潔に提示してください。
ユーザーの記述に含まれていない情報を追加で評価しないでください（記述された内容のみを対象とする）。

Phase 2: 理解度評価
Phase 1の正誤判定結果を踏まえた上で、以下の3観点を評価する。

評価観点
- 要点の網羅性
- 理解の深さ
- 説明の構造

各観点を以下の3段階で評価する:
- good
- fair
- poor

復習推奨ポイント:
- incorrect だった主張の関連トピック
- 要点漏れとして指摘した概念
- 理解が浅いと判断した箇所

出力形式
必ず以下のJSON形式のみで応答してください。JSON以外のテキストを含めないでください。

{
  "topic": "判定した学習テーマ（ユーザーが指定した場合はそれを使用）",
  "claims": [
    {
      "id": 1,
      "original": "ユーザーが書いた主張の原文",
      "verdict": "correct | incorrect | ambiguous",
      "correction": "incorrectの場合のみ、正しい情報を記載。correct/ambiguousの場合はnull",
      "reason": "判定理由を1-2文で簡潔に"
    }
  ],
  "comprehension": {
    "coverage": {
      "rating": "good | fair | poor",
      "missing_points": ["言及されていない重要な要点。なければ空配列"],
      "comment": "網羅性についてのコメント（1-2文）"
    },
    "depth": {
      "rating": "good | fair | poor",
      "shallow_areas": ["理解が浅い箇所の指摘。なければ空配列"],
      "comment": "理解の深さについてのコメント（1-2文）"
    },
    "structure": {
      "rating": "good | fair | poor",
      "comment": "説明の構造についてのコメント（1-2文）"
    },
    "review_suggestions": ["復習推奨ポイント（1〜3個）"]
  },
  "summary": {
    "total_claims": 0,
    "correct": 0,
    "incorrect": 0,
    "ambiguous": 0,
    "accuracy_rate": 0,
    "overall_feedback": "Phase 1（正誤）とPhase 2（理解度）を総合したフィードバック（2-3文）"
  }
}

注意事項
- accuracy_rate は correct / (correct + incorrect) × 100 で算出（ambiguous は除外）。小数点以下は四捨五入
- correct/incorrect が0件の場合、accuracy_rate は null とする
- 主張が1つも抽出できない場合は、claims を空配列にし、overall_feedback にその旨を記載
- 日本語で応答してください
- ユーザーの記述がどんな内容であっても、評価ルールに従って淡々と判定してください
'''

TEST_CASES = {
    'mixed_network': '''学習テーマ: TCP/IP
TCPはコネクションレス型のプロトコルで、3ウェイハンドシェイクで接続を確立する。HTTPのデフォルトポートは80番。UDPは再送制御を行わないためリアルタイム通信に向いている。今日はかなり理解できたと思う。''',
    'mostly_correct_python': '''学習テーマ: Pythonの実行方式
Pythonは通常インタプリタで実行され、ソースコードは一度バイトコードに変換される。for文や関数の仕組みを少し学んだ。''',
    'no_claims': '''今日はネットワークの勉強をした。少し難しかったけど面白かった。明日はもっと頑張りたい。''',
    'python_data_types_custom': '''【学習テーマ】Pythonのデータ型
【学習内容】
Pythonのリストはミュータブル（変更可能）で、タプルはイミュータブル（変更不可）である。
辞書はキーと値のペアを持ち、キーにはイミュータブルな型しか使えない。
setは重複を許さないコレクションで、順序を保持しない。''',
    'git_basics_custom': '''【学習テーマ】Gitの基本操作
【学習内容】
git addでステージングエリアに追加し、git commitで変更を確定する。
git pullはリモートの変更を fetchしてmergeする操作である。
git rebaseはコミット履歴を一直線にできるが、使うと危険なので本番では使わない方がいい。
git stashは変更を一時的に退避させる。stashした内容はブランチを切り替えても消えない。
conflictが起きたらgit merge --abortで必ず解消できる。''',
    'process_thread_custom': '''【学習テーマ】プロセスとスレッド
【学習内容】
プロセスはOSが管理するプログラムの実行単位で、それぞれ独立したメモリ空間を持つ。
スレッドはプロセス内の実行単位で、同じプロセスのスレッド同士はメモリを共有する。
マルチスレッドではデッドロックが起きることがある。デッドロックはスレッド同士がリソースを待ち合う状態。
コンテキストスイッチはCPUが別のプロセスに切り替える処理で、オーバーヘッドがある。''',
    'http_methods_custom': '''【学習テーマ】HTTPメソッド
【学習内容】
GETはデータの送信に使い、リクエストボディにデータを含める。
POSTはデータの取得に使い、キャッシュが効く。
PUTはリソースの部分更新に使う。
DELETEはリソースの削除に使い、冪等性がある。
PATCHはリソースの全体置換に使う。
HEADはレスポンスボディ付きでヘッダー情報を返すメソッドである。''',
    'react_intro_custom': '''【学習テーマ】React入門
【学習内容】
Reactは面白かった。コンポーネントの考え方が新鮮だった。
JSXはHTMLっぽく書けるのが便利だと思う。
useStateフックで状態管理ができる。状態が変わるとコンポーネントが再レンダリングされる。
明日はuseEffectをやりたい。
propsは親コンポーネントから子コンポーネントにデータを渡す仕組み。
Reactは最高のフレームワークだと思う。''',
    'javascript_no_claims_custom': '''【学習テーマ】JavaScript
【学習内容】
うーん、よくわからなかった。難しい。''',
    'calculus_basic_custom': '''【学習テーマ】微分の基礎
【学習内容】
微分とは関数の瞬間的な変化率を求める操作である。
f(x) = x^2 の導関数は f'(x) = 2x である。
微分と積分は逆の操作であり、これを微分積分学の基本定理という。
f(x) = sin(x) の導関数は f'(x) = -cos(x) である。
導関数が0になる点は極値の候補である。''',
    'statistics_custom': '''【学習テーマ】確率・統計
【学習内容】
標準偏差は分散の平方根である。
正規分布では平均±1σの範囲にデータの約68%が含まれる。
平均±2σには約99.7%が含まれる。
中央値は外れ値の影響を受けにくい代表値である。
相関係数は-1から1の値をとり、0に近いほど相関がない。
相関があれば因果関係がある。''',
    'mechanics_basic_custom': '''【学習テーマ】力学の基礎
【学習内容】
ニュートンの第一法則は慣性の法則で、外力が働かなければ物体は等速直線運動を続ける。
F = ma はニュートンの第三法則である。
重力加速度は地球上で約9.8 m/s^2。
運動エネルギーは 1/2 mv^2 で表される。
仕事の単位はジュール(J)で、1J = 1kg・m^2/s^2 である。
作用反作用の法則では、2つの力は同じ物体に働く。''',
    'elements_periodic_table_custom': '''【学習テーマ】元素と周期表
【学習内容】
水素の原子番号は1で、元素記号はH。
周期表は元素を原子番号順に並べたもので、メンデレーエフが最初に提唱した。
同じ族の元素は化学的性質が似ている。
希ガスは最外殻電子が安定しているため反応性が極めて低い。
鉄の元素記号はFaである。
水の分子式はH2Oで、共有結合でできている。''',
    'cell_biology_basic_custom': '''【学習テーマ】細胞の基礎
【学習内容】
細胞にはDNAを含む核がある。ただし赤血球は例外で核を持たない。
ミトコンドリアはATPを生成する細胞小器官で、独自のDNAを持つ。
光合成は葉緑体で行われ、二酸化炭素と水から酸素とグルコースを生成する。
リボソームはタンパク質を合成する場所である。
細胞膜はリン脂質の単層構造でできている。''',
    'meiji_restoration_custom': '''【学習テーマ】明治維新
【学習内容】
明治維新は1868年に始まった。大政奉還により徳川慶喜が天皇に政権を返上した。
廃藩置県により藩が廃止され、中央集権国家への移行が進んだ。
明治政府は富国強兵をスローガンに殖産興業政策を進めた。
西南戦争は西郷隆盛が明治政府に対して起こした反乱で、1877年に起きた。
大日本帝国憲法は1889年に公布され、伊藤博文が中心となって起草した。
明治維新の三傑は西郷隆盛、木戸孝允、坂本龍馬である。''',
    'french_revolution_custom': '''【学習テーマ】フランス革命
【学習内容】
フランス革命は1789年にバスティーユ牢獄の襲撃で始まった。
革命のスローガンは「自由・平等・博愛」である。
国王ルイ16世と王妃マリー・アントワネットは革命中にギロチンで処刑された。
革命後にナポレオンが皇帝に即位したのは1789年である。
人権宣言では人間の自由と平等が謳われた。
ロベスピエールは恐怖政治を行い、最終的に自分もギロチンで処刑された。''',
    'world_geography_custom': '''【学習テーマ】世界の地理
【学習内容】
世界で一番面積が大きい国はロシアで、2番目はカナダ。
世界で一番人口が多い国は中国である。
ナイル川は世界で最も長い川で、アフリカ大陸を流れている。
エベレストは世界最高峰で標高は約8849mである。
オーストラリアは世界最小の大陸であり、同時に一つの国でもある。
サハラ砂漠は南アメリカにある世界最大の砂漠である。''',
    'japanese_law_basic_custom': '''【学習テーマ】日本の法律の基礎
【学習内容】
日本国憲法は1947年に施行された。
憲法第9条は戦争の放棄と戦力の不保持を定めている。
刑法では18歳未満は死刑にならないと定められている。
民法の成年年齢は18歳である。
著作権は著作物を創作した時点で自動的に発生し、登録は不要である。
特許権も著作権と同様に自動的に発生する。''',
    'english_tenses_custom': '''【学習テーマ】英語の時制
【学習内容】
現在完了形は have + 過去分詞 で作る。
現在完了は過去の出来事が現在に影響を与えていることを表す。
"I have been to Paris" は「パリに行ったことがある」という経験を表す。
過去形と現在完了形は同じ意味で、互換性がある。
"since" は現在完了形と一緒に使い、起点を表す。
"for" は期間を表し、過去形でも現在完了形でも使える。''',
    'nutrition_basic_custom': '''【学習テーマ】栄養学の基礎
【学習内容】
三大栄養素は炭水化物、タンパク質、脂質である。
ビタミンCは水溶性ビタミンで、過剰摂取分は尿として排出される。
コラーゲンを食べると肌のコラーゲンが増える。
食物繊維は消化されないが、腸内環境を整える効果がある。
1日に必要な水分量は個人差があるが、一般的に約2リットルとされる。
卵は1日1個までにしないとコレステロールが上がる。''',
    'classical_japanese_verb_custom': '''【学習テーマ】古典文法の動詞活用
【学習内容】
古典の動詞には九種類の活用がある。
四段活用は「書かず、書きて、書く、書くとき、書けども、書け」のように活用する。
上一段活用は「見ず、見て、見る、見るとき、見れども、見よ」で、語幹と語尾の区別がない。
下二段活用の例として「受く」があり、「受けず、受けて、受く、受くるとき、受くれども、受けよ」と活用する。
カ行変格活用は「来」だけ、サ行変格活用は「す」「おはす」がある。
ナ行変格活用は「死ぬ」「行く」の二語である。''',
    'classical_auxiliary_custom': '''【学習テーマ】古典の助動詞
【学習内容】
助動詞「む」は推量・意志などの意味を持つ。
「き」は過去の助動詞で、直接体験した過去に用いる。
「けり」は過去の助動詞で、伝聞過去や詠嘆の意味がある。
打消の助動詞「ず」は未然形に接続する。
「べし」は終止形接続で、ラ変型活用語には連体形に接続する。
完了の助動詞「つ」「ぬ」は、ともに連用形接続である。
受身・尊敬・自発・可能を表す助動詞「る」「らる」は、すべて四段動詞の未然形に接続する。''',
    'classical_vocab_custom': '''【学習テーマ】古文単語の頻出語
【学習内容】
「あはれ」はしみじみとした情趣を表す言葉。
「をかし」は趣がある・興味深いという意味で、清少納言の美的理念。
「ありがたし」は現代語と同じく「感謝すべき」という意味である。
「うつくし」は古文では「かわいらしい」という意味で、現代の「美しい」とは異なる。
「なつかし」は現代語と同じく「過去を懐かしむ」という意味で使う。
「つとめて」は「早朝」という意味である。
「かなし」は「悲しい」だけでなく「愛しい」という意味もある。'''
}

selected_case = 'classical_vocab_custom'
user_input = TEST_CASES[selected_case]
print(user_input)

【学習テーマ】古文単語の頻出語
【学習内容】
「あはれ」はしみじみとした情趣を表す言葉。
「をかし」は趣がある・興味深いという意味で、清少納言の美的理念。
「ありがたし」は現代語と同じく「感謝すべき」という意味である。
「うつくし」は古文では「かわいらしい」という意味で、現代の「美しい」とは異なる。
「なつかし」は現代語と同じく「過去を懐かしむ」という意味で使う。
「つとめて」は「早朝」という意味である。
「かなし」は「悲しい」だけでなく「愛しい」という意味もある。


In [106]:
import time
from urllib.error import URLError


def post_json(url: str, headers: dict, payload: dict, timeout_seconds: int = 120) -> dict:
    req = request.Request(
        url,
        data=json.dumps(payload).encode('utf-8'),
        headers={**headers, 'Content-Type': 'application/json'},
        method='POST',
    )
    try:
        with request.urlopen(req, timeout=timeout_seconds) as response:
            return json.loads(response.read().decode('utf-8'))
    except error.HTTPError as exc:
        body = exc.read().decode('utf-8', errors='ignore')
        raise RuntimeError(f'HTTP {exc.code}: {body}') from exc
    except (TimeoutError, URLError) as exc:
        raise RuntimeError(f'Timeout or network error: {exc}') from exc


def parse_json_text(text: str):
    text = text.strip()

    if text.startswith('```'):
        lines = text.splitlines()
        if lines:
            lines = lines[1:]
        if lines and lines[-1].strip() == '```':
            lines = lines[:-1]
        text = '\n'.join(lines).strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        start = text.find('{')
        end = text.rfind('}')
        if start != -1 and end != -1 and start < end:
            try:
                return json.loads(text[start:end + 1])
            except json.JSONDecodeError:
                pass
        return {'raw_text': text}


def extract_openai_response_text(data: dict) -> str:
    if data.get('output_text'):
        return data['output_text']

    texts = []
    for item in data.get('output', []):
        for part in item.get('content', []):
            if part.get('type') in ('output_text', 'text'):
                texts.append(part.get('text', ''))
    return ''.join(texts).strip()


def call_openai(system_prompt: str, user_text: str, model: str = OPENAI_MODEL):
    headers = {'Authorization': f'Bearer {OPENAI_API_KEY}'}

    if model.startswith('gpt-5') or model.endswith('-pro') or model in ('gpt-5.4', 'gpt-5.2-pro', 'gpt-5-pro'):
        payload = {
            'model': model,
            'input': [
                {'role': 'system', 'content': system_prompt},
                {'role': 'user', 'content': user_text},
            ],
            'reasoning': {'effort': OPENAI_REASONING_EFFORT},
            'max_output_tokens': OPENAI_MAX_OUTPUT_TOKENS,
        }
        last_error = None
        for _ in range(2):
            try:
                data = post_json(
                    'https://api.openai.com/v1/responses',
                    headers=headers,
                    payload=payload,
                    timeout_seconds=300,
                )
                content = extract_openai_response_text(data)
                return {'provider': 'openai', 'model': model, 'parsed': parse_json_text(content), 'raw': content}
            except RuntimeError as exc:
                last_error = exc
                if 'Timeout or network error' in str(exc):
                    continue
                raise
        raise last_error

    payload = {
        'model': model,
        'messages': [
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_text},
        ],
        'response_format': {'type': 'json_object'},
        'max_tokens': OPENAI_MAX_OUTPUT_TOKENS,
    }
    data = post_json(
        'https://api.openai.com/v1/chat/completions',
        headers=headers,
        payload=payload,
    )
    content = data['choices'][0]['message']['content']
    return {'provider': 'openai', 'model': model, 'parsed': parse_json_text(content), 'raw': content}


def call_anthropic(system_prompt: str, user_text: str, model: str = ANTHROPIC_MODEL):
    payload = {
        'model': model,
        'max_tokens': 2000,
        'temperature': 0,
        'system': system_prompt,
        'messages': [
            {'role': 'user', 'content': user_text},
        ],
    }
    data = post_json(
        'https://api.anthropic.com/v1/messages',
        headers={
            'x-api-key': ANTHROPIC_API_KEY,
            'anthropic-version': '2023-06-01',
        },
        payload=payload,
    )
    text = ''.join(block.get('text', '') for block in data.get('content', []) if block.get('type') == 'text')
    return {'provider': 'anthropic', 'model': model, 'parsed': parse_json_text(text), 'raw': text}


def call_gemini(system_prompt: str, user_text: str, model: str = GEMINI_MODEL):
    payload = {
        'system_instruction': {
            'parts': [
                {'text': system_prompt},
            ]
        },
        'contents': [
            {
                'parts': [
                    {'text': user_text},
                ]
            }
        ],
        'generationConfig': {
            'responseMimeType': 'application/json'
        }
    }

    if GEMINI_ALLOW_FALLBACK:
        models_to_try = [model, 'gemini-2.5-pro', 'gemini-3.1-flash-lite-preview', 'gemini-3-flash-preview', 'gemini-2.5-flash']
    else:
        models_to_try = [model]

    last_error = None

    for model_name in dict.fromkeys(models_to_try):
        clean_name = model_name.replace('models/', '')
        for _ in range(3):
            try:
                data = post_json(
                    f'https://generativelanguage.googleapis.com/v1beta/models/{clean_name}:generateContent?key={GEMINI_API_KEY}',
                    headers={},
                    payload=payload,
                )
                candidates = data.get('candidates', [])
                parts = candidates[0].get('content', {}).get('parts', []) if candidates else []
                text = ''.join(part.get('text', '') for part in parts)
                return {'provider': 'gemini', 'model': clean_name, 'parsed': parse_json_text(text), 'raw': text}
            except RuntimeError as exc:
                last_error = exc
                if 'HTTP 503' in str(exc) or 'HTTP 429' in str(exc):
                    time.sleep(2)
                    continue
                if 'HTTP 404' in str(exc):
                    break
                raise

    raise last_error


def run_single_model(provider: str, user_text: str):
    if provider == 'openai':
        return call_openai(SYSTEM_PROMPT, user_text)
    if provider == 'gemini':
        return call_gemini(SYSTEM_PROMPT, user_text)
    return call_anthropic(SYSTEM_PROMPT, user_text)

In [107]:
result = run_single_model(SELECTED_PROVIDER, user_input)

print(result['provider'].upper(), '-', result['model'])
print(json.dumps(result['parsed'], ensure_ascii=False, indent=2))

OPENAI - gpt-5.4-nano
{
  "topic": "古文単語の頻出語",
  "claims": [
    {
      "id": 1,
      "original": "「あはれ」はしみじみとした情趣を表す言葉。",
      "verdict": "correct",
      "correction": null,
      "reason": "「あはれ」はしみじみとした趣・感動を表す語として用いられます。"
    },
    {
      "id": 2,
      "original": "「をかし」は趣がある・興味深いという意味で、清少納言の美的理念。",
      "verdict": "correct",
      "correction": null,
      "reason": "「をかし」は趣深い・興味深いなどの意味で、清少納言の美意識（をかし）に関わる語として説明されます。"
    },
    {
      "id": 3,
      "original": "「ありがたし」は現代語と同じく「感謝すべき」という意味である。",
      "verdict": "correct",
      "correction": null,
      "reason": "「ありがたし」は「ありがたい（感謝すべき）」の意で用いられることが一般的です。"
    },
    {
      "id": 4,
      "original": "「うつくし」は古文では「かわいらしい」という意味で、現代の「美しい」とは異なる。",
      "verdict": "correct",
      "correction": null,
      "reason": "古文の「うつくし」は「かわいらしい／いとおしい」系の意味で、「美しい」とは対応がズレます。"
    },
    {
      "id": 5,
      "original": "「なつかし」は現代語と同じく「過去を懐かしむ」という意味で使う。",
      "verdict": "ambiguous",
      "correction": null,
      "reason": "「なつかし」は「懐かし

In [108]:
CHUNK_SIZE = 6
case_items = list(TEST_CASES.items())
chunk_count = (len(case_items) + CHUNK_SIZE - 1) // CHUNK_SIZE

batch_results = {}

for chunk_index in range(chunk_count):
    start = chunk_index * CHUNK_SIZE
    end = start + CHUNK_SIZE
    chunk = case_items[start:end]

    print(f'Running chunk {chunk_index + 1}/{chunk_count} ({start + 1}-{min(end, len(case_items))})')

    for case_name, case_text in chunk:
        try:
            batch_results[case_name] = run_single_model(SELECTED_PROVIDER, case_text)
            print(f'  ✓ {case_name}')
        except Exception as exc:
            batch_results[case_name] = {
                'provider': SELECTED_PROVIDER,
                'model': OPENAI_MODEL if SELECTED_PROVIDER == 'openai' else ANTHROPIC_MODEL if SELECTED_PROVIDER == 'anthropic' else GEMINI_MODEL,
                'parsed': {},
                'raw': None,
                'error': str(exc),
            }
            print(f'  ✗ {case_name}: {exc}')

summary_view = {
    case_name: {
        'provider': data['provider'],
        'model': data.get('model'),
        'topic': data['parsed'].get('topic') if isinstance(data.get('parsed'), dict) else None,
        'total_claims': data['parsed'].get('summary', {}).get('total_claims') if isinstance(data.get('parsed'), dict) else None,
        'correct': data['parsed'].get('summary', {}).get('correct') if isinstance(data.get('parsed'), dict) else None,
        'incorrect': data['parsed'].get('summary', {}).get('incorrect') if isinstance(data.get('parsed'), dict) else None,
        'ambiguous': data['parsed'].get('summary', {}).get('ambiguous') if isinstance(data.get('parsed'), dict) else None,
        'accuracy_rate': data['parsed'].get('summary', {}).get('accuracy_rate') if isinstance(data.get('parsed'), dict) else None,
        'error': data.get('error'),
    }
    for case_name, data in batch_results.items()
}

print(json.dumps(summary_view, ensure_ascii=False, indent=2))

Running chunk 1/4 (1-6)
  ✓ mixed_network
  ✓ mostly_correct_python
  ✓ no_claims
  ✓ python_data_types_custom
  ✓ git_basics_custom
  ✓ process_thread_custom
Running chunk 2/4 (7-12)
  ✓ http_methods_custom
  ✓ react_intro_custom
  ✓ javascript_no_claims_custom
  ✓ calculus_basic_custom
  ✓ statistics_custom
  ✓ mechanics_basic_custom
Running chunk 3/4 (13-18)
  ✓ elements_periodic_table_custom
  ✓ cell_biology_basic_custom
  ✓ meiji_restoration_custom
  ✓ french_revolution_custom
  ✓ world_geography_custom
  ✓ japanese_law_basic_custom
Running chunk 4/4 (19-23)
  ✓ english_tenses_custom
  ✓ nutrition_basic_custom
  ✓ classical_japanese_verb_custom
  ✓ classical_auxiliary_custom
  ✓ classical_vocab_custom
{
  "mixed_network": {
    "provider": "openai",
    "model": "gpt-5.4-nano",
    "topic": "TCP/IP",
    "total_claims": 3,
    "correct": 1,
    "incorrect": 1,
    "ambiguous": 1,
    "accuracy_rate": 50,
    "error": null
  },
  "mostly_correct_python": {
    "provider": "openai",

In [11]:
parsed = result.get('parsed', {})
compact_view = {
    'topic': parsed.get('topic') if isinstance(parsed, dict) else None,
    'summary': parsed.get('summary') if isinstance(parsed, dict) else None,
    'claims': [
        {
            'original': claim.get('original'),
            'verdict': claim.get('verdict'),
        }
        for claim in parsed.get('claims', [])
    ] if isinstance(parsed, dict) else [],
}
print(json.dumps(compact_view, ensure_ascii=False, indent=2))

{
  "topic": "古文単語の頻出語",
  "summary": {
    "total_claims": 10,
    "correct": 8,
    "incorrect": 2,
    "ambiguous": 0,
    "accuracy_rate": 80,
    "overall_feedback": "主要な古文単語の意味をかなり押さえられており、正確性は全体として高めです。ただし、「ありがたし」「なつかし」のような現代語と意味がずれる語に誤りがあり、ここは古文読解で頻出の落とし穴です。語の基本義だけでなく、現代語との差と多義性を意識して復習すると理解が安定します。"
  },
  "claims": [
    {
      "original": "「あはれ」はしみじみとした情趣を表す言葉。",
      "verdict": "correct"
    },
    {
      "original": "「をかし」は趣がある・興味深いという意味。",
      "verdict": "correct"
    },
    {
      "original": "「をかし」は清少納言の美的理念。",
      "verdict": "correct"
    },
    {
      "original": "「ありがたし」は現代語と同じく「感謝すべき」という意味である。",
      "verdict": "incorrect"
    },
    {
      "original": "「うつくし」は古文では「かわいらしい」という意味。",
      "verdict": "correct"
    },
    {
      "original": "「うつくし」は現代の「美しい」とは異なる。",
      "verdict": "correct"
    },
    {
      "original": "「なつかし」は現代語と同じく「過去を懐かしむ」という意味で使う。",
      "verdict": "incorrect"
    },
    {
      "original": "「つとめて」は「早朝」という意味である。",
      "verdict": "co

In [185]:
models_data = request.urlopen(
    request.Request(
        f'https://generativelanguage.googleapis.com/v1beta/models?key={GEMINI_API_KEY}'
    ),
    timeout=120,
)
models_json = json.loads(models_data.read().decode('utf-8'))
flash_models = [
    {
        'name': m.get('name'),
        'displayName': m.get('displayName'),
        'supportedGenerationMethods': m.get('supportedGenerationMethods'),
    }
    for m in models_json.get('models', [])
    if 'flash' in m.get('name', '').lower()
]
print(json.dumps(flash_models, ensure_ascii=False, indent=2))

[
  {
    "name": "models/gemini-2.5-flash",
    "displayName": "Gemini 2.5 Flash",
    "supportedGenerationMethods": [
      "generateContent",
      "countTokens",
      "createCachedContent",
      "batchGenerateContent"
    ]
  },
  {
    "name": "models/gemini-2.0-flash",
    "displayName": "Gemini 2.0 Flash",
    "supportedGenerationMethods": [
      "generateContent",
      "countTokens",
      "createCachedContent",
      "batchGenerateContent"
    ]
  },
  {
    "name": "models/gemini-2.0-flash-001",
    "displayName": "Gemini 2.0 Flash 001",
    "supportedGenerationMethods": [
      "generateContent",
      "countTokens",
      "createCachedContent",
      "batchGenerateContent"
    ]
  },
  {
    "name": "models/gemini-2.0-flash-lite-001",
    "displayName": "Gemini 2.0 Flash-Lite 001",
    "supportedGenerationMethods": [
      "generateContent",
      "countTokens",
      "createCachedContent",
      "batchGenerateContent"
    ]
  },
  {
    "name": "models/gemini-2.0-flash

In [109]:
gemini_summary = {}
valid_rates = []

for case_name, data in batch_results.items():
    parsed = data.get('parsed', {}) if isinstance(data, dict) else {}
    summary = parsed.get('summary', {}) if isinstance(parsed, dict) else {}
    rate = summary.get('accuracy_rate')
    gemini_summary[case_name] = {
        'topic': parsed.get('topic'),
        'accuracy_rate': rate,
        'correct': summary.get('correct'),
        'incorrect': summary.get('incorrect'),
        'ambiguous': summary.get('ambiguous'),
        'error': data.get('error') if isinstance(data, dict) else None,
    }
    if isinstance(rate, (int, float)):
        valid_rates.append(rate)

overview = {
    'provider': SELECTED_PROVIDER,
    'model': next(iter(batch_results.values())).get('model') if batch_results else None,
    'cases': len(batch_results),
    'scored_cases': len(valid_rates),
    'average_accuracy_rate': round(sum(valid_rates) / len(valid_rates), 1) if valid_rates else None,
    'results': gemini_summary,
}

print(json.dumps(overview, ensure_ascii=False, indent=2))

{
  "provider": "openai",
  "model": "gpt-5.4-nano",
  "cases": 23,
  "scored_cases": 19,
  "average_accuracy_rate": 77.9,
  "results": {
    "mixed_network": {
      "topic": "TCP/IP",
      "accuracy_rate": 50,
      "correct": 1,
      "incorrect": 1,
      "ambiguous": 1,
      "error": null
    },
    "mostly_correct_python": {
      "topic": "Pythonの実行方式",
      "accuracy_rate": null,
      "correct": 0,
      "incorrect": 0,
      "ambiguous": 1,
      "error": null
    },
    "no_claims": {
      "topic": null,
      "accuracy_rate": null,
      "correct": 0,
      "incorrect": 0,
      "ambiguous": 0,
      "error": null
    },
    "python_data_types_custom": {
      "topic": "Pythonのデータ型",
      "accuracy_rate": 100,
      "correct": 3,
      "incorrect": 0,
      "ambiguous": 0,
      "error": null
    },
    "git_basics_custom": {
      "topic": "Gitの基本操作",
      "accuracy_rate": 75,
      "correct": 3,
      "incorrect": 1,
      "ambiguous": 1,
      "error": null
    },


In [191]:
pro_models = [
    {
        'name': m.get('name'),
        'displayName': m.get('displayName'),
        'supportedGenerationMethods': m.get('supportedGenerationMethods'),
    }
    for m in models_json.get('models', [])
    if 'pro' in m.get('name', '').lower()
]
print(json.dumps(pro_models, ensure_ascii=False, indent=2))

[
  {
    "name": "models/gemini-2.5-pro",
    "displayName": "Gemini 2.5 Pro",
    "supportedGenerationMethods": [
      "generateContent",
      "countTokens",
      "createCachedContent",
      "batchGenerateContent"
    ]
  },
  {
    "name": "models/gemini-2.5-pro-preview-tts",
    "displayName": "Gemini 2.5 Pro Preview TTS",
    "supportedGenerationMethods": [
      "countTokens",
      "generateContent",
      "batchGenerateContent"
    ]
  },
  {
    "name": "models/gemini-pro-latest",
    "displayName": "Gemini Pro Latest",
    "supportedGenerationMethods": [
      "generateContent",
      "countTokens",
      "createCachedContent",
      "batchGenerateContent"
    ]
  },
  {
    "name": "models/gemini-3-pro-preview",
    "displayName": "Gemini 3 Pro Preview",
    "supportedGenerationMethods": [
      "generateContent",
      "countTokens",
      "createCachedContent",
      "batchGenerateContent"
    ]
  },
  {
    "name": "models/gemini-3.1-pro-preview",
    "displayName": "

In [196]:
claims_only = {}

for case_name, data in batch_results.items():
    parsed = data.get('parsed', {}) if isinstance(data, dict) else {}
    claims_only[case_name] = {
        'topic': parsed.get('topic'),
        'claims': parsed.get('claims', []) if isinstance(parsed, dict) else [],
        'error': data.get('error') if isinstance(data, dict) else None,
    }

print(json.dumps(claims_only, ensure_ascii=False, indent=2))

{
  "mixed_network": {
    "topic": "TCP/IP",
    "claims": [
      {
        "id": 1,
        "original": "TCPはコネクションレス型のプロトコルである",
        "verdict": "incorrect",
        "correction": "TCPはコネクション型のプロトコルです。",
        "reason": "TCPはデータ転送前に通信相手との論理的な経路（コネクション）を確立するコネクション型のプロトコルです。コネクションレス型はUDPが該当します。"
      },
      {
        "id": 2,
        "original": "TCPは3ウェイハンドシェイクで接続を確立する",
        "verdict": "correct",
        "correction": null,
        "reason": "TCPは通信を開始する際、SYN、SYN/ACK、ACKの3回のパケット交換により接続を確立します。"
      },
      {
        "id": 3,
        "original": "HTTPのデフォルトポートは80番である",
        "verdict": "correct",
        "correction": null,
        "reason": "HTTPプロトコルのウェルノウンポート（デフォルト）は80番として割り当てられています。"
      },
      {
        "id": 4,
        "original": "UDPは再送制御を行わないためリアルタイム通信に向いている",
        "verdict": "correct",
        "correction": null,
        "reason": "UDPは再送制御や順序制御を省くことでオーバーヘッドを小さくしており、遅延が重視されるリアルタイム通信に適しています。"
      }
    ],
    "error": null
  },
  "mostly_correct_pyth

In [197]:
claims_verdict_only = {}

for case_name, data in batch_results.items():
    parsed = data.get('parsed', {}) if isinstance(data, dict) else {}
    claims = parsed.get('claims', []) if isinstance(parsed, dict) else []
    claims_verdict_only[case_name] = {
        'topic': parsed.get('topic'),
        'claims': [
            {
                'original': claim.get('original'),
                'verdict': claim.get('verdict'),
            }
            for claim in claims
        ],
        'error': data.get('error') if isinstance(data, dict) else None,
    }

print(json.dumps(claims_verdict_only, ensure_ascii=False, indent=2))

{
  "mixed_network": {
    "topic": "TCP/IP",
    "claims": [
      {
        "original": "TCPはコネクションレス型のプロトコルである",
        "verdict": "incorrect"
      },
      {
        "original": "TCPは3ウェイハンドシェイクで接続を確立する",
        "verdict": "correct"
      },
      {
        "original": "HTTPのデフォルトポートは80番である",
        "verdict": "correct"
      },
      {
        "original": "UDPは再送制御を行わないためリアルタイム通信に向いている",
        "verdict": "correct"
      }
    ],
    "error": null
  },
  "mostly_correct_python": {
    "topic": "Pythonの実行方式",
    "claims": [
      {
        "original": "Pythonは通常インタプリタで実行される",
        "verdict": "correct"
      },
      {
        "original": "ソースコードは一度バイトコードに変換される",
        "verdict": "correct"
      }
    ],
    "error": null
  },
  "no_claims": {
    "topic": "ネットワーク",
    "claims": [],
    "error": null
  },
  "python_data_types_custom": {
    "topic": "Pythonのデータ型",
    "claims": [
      {
        "original": "Pythonのリストはミュータブル（変更可能）である。",
        "verdict": "correct"
    

In [110]:
from pathlib import Path

verdict_map = {
    'correct': '正',
    'incorrect': '誤',
    'ambiguous': '曖昧',
}

model_name = batch_results[next(iter(batch_results))].get('model') if batch_results else 'unknown-model'
safe_model_name = str(model_name).replace('/', '_').replace(':', '_')

lines = []
for idx, (case_name, data) in enumerate(batch_results.items(), 1):
    parsed = data.get('parsed', {}) if isinstance(data, dict) else {}
    topic = parsed.get('topic') or 'topicなし'
    claims = parsed.get('claims', []) if isinstance(parsed, dict) else []

    lines.append(f'{idx}. {case_name} ／ {topic}')

    if data.get('error'):
        lines.append(f'エラー: {data.get("error")}')
    elif not claims:
        lines.append('claimsなし')
    else:
        for claim in claims:
            verdict = verdict_map.get(claim.get('verdict'), claim.get('verdict'))
            lines.append(f'{claim.get("original")} → {verdict}')

    lines.append('')

summary_path = Path.cwd() / f'{safe_model_name}_claims_by_test.md'
summary_path.write_text('\n'.join(lines), encoding='utf-8')
print(summary_path)
print('\n'.join(lines[:160]))

c:\Users\Admin\Desktop\研究関連\putokei\Hourglass\experiement\gpt-5.4-nano_claims_by_test.md
1. mixed_network ／ TCP/IP
TCPはコネクションレス型のプロトコルで、3ウェイハンドシェイクで接続を確立する。 → 誤
HTTPのデフォルトポートは80番。 → 正
UDPは再送制御を行わないためリアルタイム通信に向いている。 → 曖昧

2. mostly_correct_python ／ Pythonの実行方式
Pythonは通常インタプリタで実行され、ソースコードは一度バイトコードに変換される。 → 曖昧

3. no_claims ／ topicなし
claimsなし

4. python_data_types_custom ／ Pythonのデータ型
Pythonのリストはミュータブル（変更可能）で、タプルはイミュータブル（変更不可）である。 → 正
辞書はキーと値のペアを持ち、キーにはイミュータブルな型しか使えない。 → 正
setは重複を許さないコレクションで、順序を保持しない。 → 正

5. git_basics_custom ／ Gitの基本操作
git addでステージングエリアに追加し、git commitで変更を確定する。 → 正
git pullはリモートの変更を fetchしてmergeする操作である。 → 曖昧
git rebaseはコミット履歴を一直線にできるが、 → 正
stashした内容はブランチを切り替えても消えない。 → 正
conflictが起きたらgit merge --abortで必ず解消できる。 → 誤

6. process_thread_custom ／ プロセスとスレッド
プロセスはOSが管理するプログラムの実行単位で、それぞれ独立したメモリ空間を持つ。 → 正
スレッドはプロセス内の実行単位で、同じプロセスのスレッド同士はメモリを共有する。 → 正
マルチスレッドではデッドロックが起きることがある。 → 正
デッドロックはスレッド同士がリソースを待ち合う状態。 → 正
コンテキストスイッチはCPUが別のプロセスに切り替える処理で、オーバーヘッドがある。 → 正

7. http_methods_custom ／

In [97]:
req = request.Request(
    'https://api.openai.com/v1/models',
    headers={'Authorization': f'Bearer {OPENAI_API_KEY}'},
    method='GET',
)
with request.urlopen(req, timeout=120) as response:
    openai_models = json.loads(response.read().decode('utf-8')).get('data', [])

openai_gpt5_models = sorted(
    [m.get('id') for m in openai_models if 'gpt-5' in m.get('id', '') or 'chatgpt' in m.get('id', '')]
)
print(json.dumps(openai_gpt5_models, ensure_ascii=False, indent=2))

[
  "chatgpt-image-latest",
  "gpt-5",
  "gpt-5-2025-08-07",
  "gpt-5-chat-latest",
  "gpt-5-codex",
  "gpt-5-mini",
  "gpt-5-mini-2025-08-07",
  "gpt-5-nano",
  "gpt-5-nano-2025-08-07",
  "gpt-5-pro",
  "gpt-5-pro-2025-10-06",
  "gpt-5-search-api",
  "gpt-5-search-api-2025-10-14",
  "gpt-5.1",
  "gpt-5.1-2025-11-13",
  "gpt-5.1-chat-latest",
  "gpt-5.1-codex",
  "gpt-5.1-codex-max",
  "gpt-5.1-codex-mini",
  "gpt-5.2",
  "gpt-5.2-2025-12-11",
  "gpt-5.2-chat-latest",
  "gpt-5.2-codex",
  "gpt-5.2-pro",
  "gpt-5.2-pro-2025-12-11",
  "gpt-5.3-chat-latest",
  "gpt-5.3-codex",
  "gpt-5.4",
  "gpt-5.4-2026-03-05",
  "gpt-5.4-mini",
  "gpt-5.4-mini-2026-03-17",
  "gpt-5.4-nano",
  "gpt-5.4-nano-2026-03-17",
  "gpt-5.4-pro",
  "gpt-5.4-pro-2026-03-05"
]


In [111]:
from pathlib import Path

rates = []
summary_lines = [f'# {SELECTED_PROVIDER} / {batch_results[next(iter(batch_results))].get("model") if batch_results else None}']
summary_lines.append('')

for case_name, data in batch_results.items():
    parsed = data.get('parsed', {}) if isinstance(data, dict) else {}
    summary = parsed.get('summary', {}) if isinstance(parsed, dict) else {}
    rate = summary.get('accuracy_rate')
    if isinstance(rate, (int, float)):
        rates.append(rate)
    summary_lines.append(f'- {case_name}: {rate}')

summary_lines.append('')
summary_lines.append(f'average_accuracy_rate: {round(sum(rates) / len(rates), 1) if rates else None}')

summary_file = Path.cwd() / 'current_model_summary.md'
summary_file.write_text('\n'.join(summary_lines), encoding='utf-8')
print(summary_file)
print('\n'.join(summary_lines))

c:\Users\Admin\Desktop\研究関連\putokei\Hourglass\experiement\current_model_summary.md
# openai / gpt-5.4-nano

- mixed_network: 50
- mostly_correct_python: None
- no_claims: None
- python_data_types_custom: 100
- git_basics_custom: 75
- process_thread_custom: 100
- http_methods_custom: 20
- react_intro_custom: 100
- javascript_no_claims_custom: None
- calculus_basic_custom: 75
- statistics_custom: 80
- mechanics_basic_custom: 67
- elements_periodic_table_custom: 83
- cell_biology_basic_custom: 83
- meiji_restoration_custom: 86
- french_revolution_custom: 83
- world_geography_custom: 67
- japanese_law_basic_custom: 83
- english_tenses_custom: 83
- nutrition_basic_custom: 60
- classical_japanese_verb_custom: None
- classical_auxiliary_custom: 86
- classical_vocab_custom: 100

average_accuracy_rate: 77.9


In [36]:
candidate_models = [
    'claude-haiku-4-5',
    'claude-3-haiku-20240307',
    'claude-3-5-haiku-20241022',
    'claude-3-5-haiku-latest',
    'claude-haiku-4-20250514',
    'claude-haiku-4-latest',
    'claude-haiku-4-5-latest',
]

availability = {}
for model_name in candidate_models:
    try:
        test = call_anthropic(SYSTEM_PROMPT, user_input, model=model_name)
        availability[model_name] = {'ok': True, 'model': test.get('model')}
    except Exception as exc:
        availability[model_name] = {'ok': False, 'error': str(exc)}

print(json.dumps(availability, ensure_ascii=False, indent=2))

{
  "claude-haiku-4-5": {
    "ok": true,
    "model": "claude-haiku-4-5"
  },
  "claude-3-haiku-20240307": {
    "ok": true,
    "model": "claude-3-haiku-20240307"
  },
  "claude-3-5-haiku-20241022": {
    "ok": false,
    "error": "HTTP 404: {\"type\":\"error\",\"error\":{\"type\":\"not_found_error\",\"message\":\"model: claude-3-5-haiku-20241022\"},\"request_id\":\"req_011Ca9BMuiJr9ovQqCfzB188\"}"
  },
  "claude-3-5-haiku-latest": {
    "ok": false,
    "error": "HTTP 404: {\"type\":\"error\",\"error\":{\"type\":\"not_found_error\",\"message\":\"model: claude-3-5-haiku-latest\"},\"request_id\":\"req_011Ca9BMwCMNsqua7UMhkwXM\"}"
  },
  "claude-haiku-4-20250514": {
    "ok": false,
    "error": "HTTP 404: {\"type\":\"error\",\"error\":{\"type\":\"not_found_error\",\"message\":\"model: claude-haiku-4-20250514\"},\"request_id\":\"req_011Ca9BMxoLm3NtrFegKKDdo\"}"
  },
  "claude-haiku-4-latest": {
    "ok": false,
    "error": "HTTP 404: {\"type\":\"error\",\"error\":{\"type\":\"not_found